# Meteologica Wind Forecast — DA Cutoff Revision Analysis (RTO)

Inspects how Meteologica PJM **RTO-level wind generation** forecasts for the next 7 days evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 12h / 24h / 48h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates.

## 1. Setup & Data Pull

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
sql_path = Path.cwd() / "meteologica_wind_forecast_da_cutoff_rto.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"Region: {df['region'].unique()}")
df.head(10)

186 rows
Date range: 2026-03-04 to 2026-03-11
Region: <ArrowStringArray>
['RTO']
Length: 1, dtype: str


,forecast_date,hour_ending,region,cutoff_gen_mw,cutoff_execution_dt,exec_dt_12h,gen_mw_12h,exec_dt_24h,gen_mw_24h,exec_dt_48h,gen_mw_48h,delta_12h,delta_24h,delta_48h
0,2026-03-11,1,RTO,7600.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7900.0,2026-03-03 07:27:49,7568.0,2026-02-27 12:23:55,5340.0,-300.0,32.0,2260.0
1,2026-03-11,2,RTO,7465.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7724.0,2026-03-03 07:27:49,7421.0,2026-02-27 12:23:55,5226.0,-259.0,44.0,2239.0
2,2026-03-11,3,RTO,7635.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7748.0,2026-03-03 07:27:49,7489.0,2026-02-27 12:23:55,5171.0,-113.0,146.0,2464.0
3,2026-03-11,4,RTO,7935.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7886.0,2026-03-03 07:27:49,7670.0,2026-02-27 12:23:55,5159.0,49.0,265.0,2776.0
4,2026-03-11,5,RTO,8117.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7961.0,2026-03-03 07:27:49,7745.0,2026-02-27 12:23:55,5129.0,156.0,372.0,2988.0
5,2026-03-11,6,RTO,8138.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7921.0,2026-03-03 07:27:49,7649.0,2026-02-27 12:23:55,5058.0,217.0,489.0,3080.0
6,2026-03-11,7,RTO,8049.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7793.0,2026-03-03 07:27:49,7455.0,2026-02-27 12:23:55,4990.0,256.0,594.0,3059.0
7,2026-03-11,8,RTO,7951.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7660.0,2026-03-03 07:27:49,7339.0,2026-02-27 12:23:55,4961.0,291.0,612.0,2990.0
8,2026-03-11,9,RTO,7942.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7672.0,2026-03-03 07:27:49,7396.0,2026-02-27 12:23:55,4975.0,270.0,546.0,2967.0
9,2026-03-11,10,RTO,8020.0,2026-03-04 08:46:59,2026-03-03 19:25:49,7785.0,2026-03-03 07:27:49,7538.0,2026-02-27 12:23:55,5006.0,235.0,482.0,3014.0


In [3]:
df.dtypes

forecast_date                  object
hour_ending                     int64
region                            str
cutoff_gen_mw                 float64
cutoff_execution_dt    datetime64[us]
exec_dt_12h            datetime64[us]
gen_mw_12h                    float64
exec_dt_24h            datetime64[us]
gen_mw_24h                    float64
exec_dt_48h            datetime64[us]
gen_mw_48h                    float64
delta_12h                     float64
delta_24h                     float64
delta_48h                     float64
dtype: object

## 1b. Forecast Coverage Summary

One row per forecast date showing the cutoff vintage used, each lookback vintage,
and peak cutoff generation — a quick reference for which days have full 48h→12h history.

In [4]:
_fmt_dt = lambda s: pd.to_datetime(s).strftime("%Y-%m-%d %H:%M") if pd.notna(s) else "\u2014"

forecast_summary = (
    df.groupby("forecast_date")
    .agg(
        cutoff_exec=("cutoff_execution_dt", "first"),
        exec_12h=("exec_dt_12h", "first"),
        exec_24h=("exec_dt_24h", "first"),
        exec_48h=("exec_dt_48h", "first"),
        peak_cutoff_mw=("cutoff_gen_mw", "max"),
        min_cutoff_mw=("cutoff_gen_mw", "min"),
        hours_covered=("hour_ending", "nunique"),
    )
    .reset_index()
    .sort_values("forecast_date")
)

forecast_summary["days_ahead"] = (
    pd.to_datetime(forecast_summary["forecast_date"]) - pd.Timestamp.now().normalize()
).dt.days

for col in ["cutoff_exec", "exec_12h", "exec_24h", "exec_48h"]:
    forecast_summary[col] = forecast_summary[col].apply(_fmt_dt)

forecast_summary["has_12h"] = forecast_summary["exec_12h"] != "\u2014"
forecast_summary["has_24h"] = forecast_summary["exec_24h"] != "\u2014"
forecast_summary["has_48h"] = forecast_summary["exec_48h"] != "\u2014"

display_cols = [
    "forecast_date", "days_ahead", "hours_covered",
    "cutoff_exec", "exec_48h", "exec_24h", "exec_12h",
    "has_48h", "has_24h", "has_12h",
    "peak_cutoff_mw", "min_cutoff_mw",
]
forecast_summary[display_cols].style.set_caption(
    "Meteologica RTO Wind \u2014 Forecast Date Coverage \u2014 7-Day Forward Window"
).format({
    "peak_cutoff_mw": "{:,.0f}",
    "min_cutoff_mw": "{:,.0f}",
}).set_properties(**{"text-align": "center"})

,forecast_date,days_ahead,hours_covered,cutoff_exec,exec_48h,exec_24h,exec_12h,has_48h,has_24h,has_12h,peak_cutoff_mw,min_cutoff_mw
0,2026-03-04,0,20,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"2,576",754
1,2026-03-05,1,24,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"3,161","2,130"
2,2026-03-06,2,24,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"8,363","2,956"
3,2026-03-07,3,23,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"8,338","3,673"
4,2026-03-08,4,23,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"7,885","3,841"
5,2026-03-09,5,24,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"7,739","5,741"
6,2026-03-10,6,24,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"8,045","7,380"
7,2026-03-11,7,24,2026-03-04 08:46,2026-02-27 12:23,2026-03-03 07:27,2026-03-03 19:25,True,True,True,"8,138","5,878"


## 2. Data Validation — Cutoff Vintage Inspection

In [5]:
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-03-04,2026-03-04 08:46:59,08:46
1,2026-03-05,2026-03-04 08:46:59,08:46
2,2026-03-06,2026-03-04 08:46:59,08:46
3,2026-03-07,2026-03-04 08:46:59,08:46
4,2026-03-08,2026-03-04 08:46:59,08:46
5,2026-03-09,2026-03-04 08:46:59,08:46
6,2026-03-10,2026-03-04 08:46:59,08:46
7,2026-03-11,2026-03-04 08:46:59,08:46


In [6]:
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Meteologica RTO Wind \u2014 Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [7]:
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("gen_mw_12h", lambda x: x.notna().any()),
        has_24h=("gen_mw_24h", lambda x: x.notna().any()),
        has_48h=("gen_mw_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-03-04,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
1,2026-03-05,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
2,2026-03-06,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
3,2026-03-07,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
4,2026-03-08,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
5,2026-03-09,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
6,2026-03-10,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55
7,2026-03-11,True,True,True,2026-03-03 19:25:49,2026-03-03 07:27:49,2026-02-27 12:23:55


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [8]:
# Overlay 48h -> 24h -> 12h -> cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].sort_values("hour_ending")

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

fig = go.Figure()
for label, col, width in [
    ("48h", "gen_mw_48h", 1),
    ("24h", "gen_mw_24h", 1.5),
    ("12h", "gen_mw_12h", 2),
    ("Cutoff", "cutoff_gen_mw", 3),
]:
    fig.add_trace(go.Scatter(
        x=df_latest["hour_ending"],
        y=df_latest[col],
        name=label,
        line=dict(color=colors[label], dash=dashes[label], width=width),
    ))

fig.update_layout(
    height=500,
    template="plotly_dark",
    title_text=f"Meteologica RTO Wind \u2014 Forecast Evolution \u2014 {latest_date}",
    xaxis_title="Hour Ending",
    yaxis_title="Generation (MW)",
)
fig.show()

In [9]:
# All dates — cutoff (solid) vs 24h lookback (dashed)
dates = sorted(df["forecast_date"].unique())

fig = go.Figure()
for d in dates:
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")
    opacity = 0.3 if d != latest_date else 1.0
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["cutoff_gen_mw"],
        name=f"{d} cutoff",
        line=dict(width=2 if d == latest_date else 1),
        opacity=opacity,
    ))
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["gen_mw_24h"],
        name=f"{d} 24h",
        line=dict(dash="dash", width=1),
        opacity=opacity * 0.6,
        showlegend=False,
    ))

fig.update_layout(
    height=500, template="plotly_dark",
    title="Meteologica RTO Wind \u2014 Cutoff (solid) vs 24h Prior (dashed)",
    xaxis_title="Hour Ending", yaxis_title="Generation (MW)",
)
fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [10]:
# Grouped bar: delta_12h, delta_24h, delta_48h by hour_ending for latest date
fig = go.Figure()
for col, label, color in [
    ("delta_48h", "48h\u2192Cutoff", "#636EFA"),
    ("delta_24h", "24h\u2192Cutoff", "#EF553B"),
    ("delta_12h", "12h\u2192Cutoff", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=df_latest["hour_ending"], y=df_latest[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title=f"Meteologica RTO Wind \u2014 Forecast Revision Deltas ({latest_date})",
    xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
    template="plotly_dark", height=450,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [11]:
# Heatmap: 24h delta across all dates x hours
pivot_24h = df.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_24h",
)

fig = px.imshow(
    pivot_24h.values,
    x=[f"HE{int(c)}" for c in pivot_24h.columns],
    y=[str(d) for d in pivot_24h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Meteologica RTO Wind \u2014 24h Forecast Revision (MW) by Date \u00d7 Hour",
    labels={"color": "Delta MW"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. Forecast Evolution by Date — All Vintages

In [12]:
dates = sorted(df["forecast_date"].unique())

fig = make_subplots(
    rows=len(dates), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"RTO Wind \u2014 {d}" for d in dates],
    vertical_spacing=0.04,
)

for i, d in enumerate(dates, 1):
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")

    for label, col, exec_col, width in [
        ("48h", "gen_mw_48h", "exec_dt_48h", 1),
        ("24h", "gen_mw_24h", "exec_dt_24h", 1.5),
        ("12h", "gen_mw_12h", "exec_dt_12h", 2),
        ("Cutoff", "cutoff_gen_mw", "cutoff_execution_dt", 3),
    ]:
        exec_dts = ddf[exec_col].astype(str).values
        forecast_dates = ddf["forecast_date"].astype(str).values
        deltas = np.zeros(len(ddf)) if label == "Cutoff" else ddf[f"delta_{label}"].values
        rank_labels = np.array([label] * len(ddf))

        customdata = np.column_stack([exec_dts, forecast_dates, deltas, rank_labels])

        fig.add_trace(
            go.Scatter(
                x=ddf["hour_ending"],
                y=ddf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[3]}</b> \u2014 RTO Wind<br>"
                    "Forecast Date: %{customdata[1]}<br>"
                    "HE: %{x}<br>"
                    "Generation: %{y:,.0f} MW<br>"
                    "Execution DT: %{customdata[0]}<br>"
                    "Delta to Cutoff: %{customdata[2]} MW"
                    "<extra></extra>"
                ),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=250 * len(dates),
    template="plotly_dark",
    title_text="Meteologica RTO Wind \u2014 Forecast Evolution by Date",
)
fig.update_yaxes(title_text="Generation (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
fig.show()

## 6. Revision Summary Statistics

In [13]:
summary_rows = []
for lookback, col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    vals = df[col].dropna()
    summary_rows.append({
        "Lookback": lookback,
        "Mean (MW)": vals.mean(),
        "Median (MW)": vals.median(),
        "Std (MW)": vals.std(),
        "Min (MW)": vals.min(),
        "Max (MW)": vals.max(),
        "N": len(vals),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Lookback,Mean (MW),Median (MW),Std (MW),Min (MW),Max (MW),N
0,12h,138.215054,117.0,469.412292,-746.0,1267.0,186
1,24h,222.016129,122.5,735.473905,-947.0,2646.0,186
2,48h,707.311828,1809.5,2302.260060,-3988.0,3557.0,186


In [14]:
# Per-date summary: avg revision direction across all hours
date_summary = (
    df.groupby("forecast_date")
    .agg(
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        mean_delta_48h=("delta_48h", "mean"),
        peak_cutoff_mw=("cutoff_gen_mw", "max"),
    )
    .reset_index()
)

fig = px.bar(
    date_summary,
    x="forecast_date",
    y="mean_delta_24h",
    title="Meteologica RTO Wind \u2014 Mean 24h Forecast Revision (MW) by Date",
    labels={"mean_delta_24h": "Avg 24h Delta (MW)", "forecast_date": "Forecast Date"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.update_layout(height=450)
fig.show()

In [15]:
# Heatmaps for all lookbacks (12h, 24h, 48h)
for lookback, delta_col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    pivot = df.pivot_table(
        index="forecast_date", columns="hour_ending", values=delta_col,
    )
    if pivot.empty:
        continue

    fig = px.imshow(
        pivot.values,
        x=[f"HE{int(c)}" for c in pivot.columns],
        y=[str(d) for d in pivot.index],
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"Meteologica RTO Wind \u2014 {lookback} Forecast Revision (MW) by Date \u00d7 Hour",
        labels={"color": "Delta MW"},
        template="plotly_dark",
    )
    fig.update_layout(height=400)
    fig.show()